In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# ==============================
# 1. LOAD DATA
# ==============================
df = pd.read_csv("../data/online_retail_customer_churn.csv")

print("Columns in dataset:")
print(df.columns)

Columns in dataset:
Index(['recency', 'frequency', 'monetary', 'avg_spend_per_purchase',
       'customer_value_score', 'recency_frequency_ratio', 'monetary_log',
       'Numeric_Mean', 'recency_x_frequency'],
      dtype='object')


In [3]:
def handle_missing_values(df):
    # Numerical columns
    num_cols = df.select_dtypes(include=np.number).columns
    num_imputer = SimpleImputer(strategy='median')
    df[num_cols] = num_imputer.fit_transform(df[num_cols])

    # Categorical columns
    cat_cols = df.select_dtypes(include='object').columns
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

    return df


def remove_outliers_iqr(df, cols):
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df[col] = np.where(df[col] < lower, lower, df[col])
        df[col] = np.where(df[col] > upper, upper, df[col])

    return df
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include='object').columns

if len(num_cols) > 0:
    num_imputer = SimpleImputer(strategy='median')
    df[num_cols] = num_imputer.fit_transform(df[num_cols])

if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

In [4]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.clip(df[col], lower, upper)

In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
def scale_features(df, method="standard"):
    scaler = StandardScaler() if method == "standard" else MinMaxScaler()
    scaled = scaler.fit_transform(df)
    df_scaled = pd.DataFrame(scaled, columns=df.columns)
    return df_scaled, scaler

def encode_categorical(df):
    df = pd.get_dummies(df, drop_first=True)
    return df
df = pd.read_csv("../data/online_retail_customer_churn.csv")
df_encoded = encode_categorical(df)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)

X_scaled = pd.DataFrame(X_scaled, columns=df_encoded.columns)

print("Final shape:", X_scaled.shape)
print("Preprocessing Completed Successfully ✅")

Final shape: (789, 9)
Preprocessing Completed Successfully ✅


In [27]:
def encode_categorical(df):
    df = pd.get_dummies(df, drop_first=True)
    return df
df_encoded = pd.get_dummies(df, drop_first=True)

# Drop ID column if exists
for col in ['CustomerID', 'RowNumber', 'Surname']:
    if col in df_encoded.columns:
        df_encoded.drop(columns=[col], inplace=True)



In [17]:

if 'Tenure' in df.columns:
    df['Recency'] = df['Tenure'].max() - df['Tenure']

if 'NumOfProducts' in df.columns:
    df['Frequency'] = df['NumOfProducts']

if 'Balance' in df.columns:
    df['Monetary'] = df['Balance']

if all(col in df.columns for col in ['Balance', 'NumOfProducts']):
    df['Balance_per_Product'] = df['Balance'] / (df['NumOfProducts'] + 1)

if all(col in df.columns for col in ['EstimatedSalary', 'Balance']):
    df['Salary_Balance_Ratio'] = df['EstimatedSalary'] / (df['Balance'] + 1)

if all(col in df.columns for col in ['Tenure', 'Age']):
    df['Tenure_Age_Ratio'] = df['Tenure'] / (df['Age'] + 1)

if all(col in df.columns for col in ['NumOfProducts', 'Tenure']):
    df['Product_per_Tenure'] = df['NumOfProducts'] / (df['Tenure'] + 1)

if all(col in df.columns for col in ['IsActiveMember', 'Balance']):
    df['Activity_Balance_Ratio'] = df['IsActiveMember'] * df['Balance']

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime


def create_rfm_features(df,
                        customer_id_col='CustomerID',
                        date_col='InvoiceDate',
                        amount_col='TotalAmount'):

    df[date_col] = pd.to_datetime(df[date_col])
    snapshot_date = df[date_col].max() + pd.Timedelta(days=1)

    rfm = df.groupby(customer_id_col).agg({
        date_col: lambda x: (snapshot_date - x.max()).days,
        customer_id_col: 'count',
        amount_col: 'sum'
    })

    rfm.columns = ['Recency', 'Frequency', 'Monetary']

    return rfm.reset_index()


def create_behavioral_features(df, customer_id_col='CustomerID'):
    behavior = df.groupby(customer_id_col).agg({
        'Quantity': ['mean', 'sum'],
        'TotalAmount': ['mean', 'max']
    })

    behavior.columns = ['Avg_Quantity', 'Total_Quantity',
                        'Avg_Spending', 'Max_Spending']

    return behavior.reset_index()


def create_derived_ratios(df):
    df['Avg_Order_Value'] = df['Monetary'] / (df['Frequency'] + 1)
    df['Purchase_Recency_Ratio'] = df['Frequency'] / (df['Recency'] + 1)
    df['Spending_Frequency_Ratio'] = df['Monetary'] / (df['Frequency'] + 1)

    return df

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer


def load_data(path):
    df = pd.read_csv(path)
    return df

In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from scipy import stats

In [8]:
df = pd.read_csv("../data/online_retail_customer_churn.csv")

print(df.head())
print(df.info())

   recency  frequency  monetary  avg_spend_per_purchase  customer_value_score  \
0        5         22   5892.58              256.199130             129636.76   
1       13         77   9025.47              115.711154             694961.19   
2       13         71    618.83                8.594861              43936.93   
3        3         33   9110.30              267.950000             300639.90   
4       15         43   5390.88              122.520000             231807.84   

   recency_frequency_ratio  monetary_log   Numeric_Mean  recency_x_frequency  
0                 0.217391      8.681619   19403.062592                  110  
1                 0.166667      9.107917  100600.235105                 1001  
2                 0.180556      6.429445    6379.280695                  923  
3                 0.088235      9.117271   44294.765072                   99  
4                 0.340909      8.592649   33912.596223                  645  
<class 'pandas.core.frame.DataFrame'>
R

In [9]:
# Check missing values
print(df.isnull().sum())

# Fill numerical columns with median
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical columns with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

recency                    0
frequency                  0
monetary                   0
avg_spend_per_purchase     0
customer_value_score       0
recency_frequency_ratio    0
monetary_log               0
Numeric_Mean               0
recency_x_frequency        0
dtype: int64


In [15]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]